# Jina Reranker Token tr?n Colab

Notebook n?y ch?y `src/re-ranker/jina_rerank_token.py` v?i model `jinaai/jina-reranker-v2-base-multilingual` tr?n input token.


In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Chưa bật GPU")

CUDA available: True
GPU: Tesla T4


In [4]:
!pip uninstall -y transformers tokenizers
!pip install -q "transformers==4.44.2" "tokenizers==0.19.1" accelerate sentence-transformers einops

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
%cd /content

!rm -rf Text-Mining---RAG-on-News
!git clone -b bge_rerank_TuanAnh https://github.com/TiiAyyLuvBear/Text-Mining---RAG-on-News.git

%cd /content/Text-Mining---RAG-on-News

/content
Cloning into 'Text-Mining---RAG-on-News'...
remote: Enumerating objects: 901, done.
remote: Counting objects: 100% (557/557), done.
remote: Compressing objects: 100% (300/300), done.
remote: Total 901 (delta 329), reused 471 (delta 253), pack-reused 344 (from 1)
Receiving objects: 100% (901/901), 26.42 MiB | 4.51 MiB/s, done.
Resolving deltas: 100% (503/503), done.
Updating files: 100% (104/104), done.
/content/Text-Mining---RAG-on-News


In [7]:
import torch
import transformers
import importlib.util

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("transformers:", transformers.__version__)
print("Jina reranker dependencies ready")

CUDA available: True
GPU: Tesla T4
transformers: 4.44.2
Jina reranker dependencies ready


In [8]:
from pathlib import Path

input_path = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_embedding/per_query_token.jsonl")

print("Input exists:", input_path.exists())
print("Input path:", input_path)

if input_path.exists():
    with input_path.open("r", encoding="utf-8") as f:
        first_line = f.readline()
    print(first_line[:500])

Input exists: True
Input path: /content/drive/MyDrive/HỌC TẬP/Text mining/out_embedding/per_query_token.jsonl
{"qa_id": "211640_1", "qa_type": "factoid", "question": "Những loại nội tạng động vật nào được khuyến cáo nên hạn chế để tránh tăng axit uric và hại thận?", "gold_articles": ["211640"], "top_articles": ["211640", "211640", "27887", "205401", "205401", "28856", "27606", "27887", "27887", "31222"], "candidates": [{"rank": 1, "chunk_index": 1, "chunk_id": "211640_token_0001", "article_id": "211640", "score": 0.680461, "text": "Tiêu đề: Không muốn hại thận, cần hạn chế 4 loại thịt\nMô tả: Purin tron


In [9]:
from pathlib import Path

script_path = Path("src/re-ranker/JINA_TOKEN/jina_rerank_token.py")

print("Script exists:", script_path.exists())
print("Script path:", script_path)


Script exists: True
Script path: src/re-ranker/JINA_TOKEN/jina_rerank_token.py


In [10]:
from pathlib import Path

output_dir = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker")
output_dir.mkdir(parents=True, exist_ok=True)

print("Output dir exists:", output_dir.exists())

Output dir exists: True


## Hugging Face

Model Jina public th??ng kh?ng c?n login. N?u g?p l?i gated/private, ch?y `from huggingface_hub import login; login()`.


In [ ]:
from huggingface_hub import login

login("hf_................")

In [17]:
!python src/re-ranker/JINA_TOKEN/jina_rerank_token.py \
    --input "/content/drive/MyDrive/HỌC TẬP/Text mining/out_embedding/per_query_token.jsonl" \
    --output "/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina_top5_test.jsonl" \
    --limit 5 \
    --batch-size 8 \
    --max-length 1024

--> Đang nạp Tokenizer cho chiến thuật Token...
--> Đang nạp Jina Reranker V2 với cấu hình ép kiểu dữ liệu nửa chính xác float16...
--> Khởi tạo hoàn tất! Thiết bị xử lý hiện hành: CUDA
Done. Wrote 5 rows to /content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina_top5_test.jsonl


In [18]:
import json
from pathlib import Path

test_output = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina_top5_test.jsonl")

print("Test output exists:", test_output.exists())

with test_output.open("r", encoding="utf-8") as f:
    row = json.loads(next(f))

print("QA ID:", row["qa_id"])
print("Question:", row["question"])
print("Embedding type:", row["embedding_type"])
print("Reranker:", row["reranker"])
print("Metrics:", row["rerank_metrics"])
print("Top 1 score:", row["reranked_candidates"][0]["rerank_score"])
print("Top 1 text:", row["reranked_candidates"][0]["text"][:300])

Test output exists: True
QA ID: 211640_1
Question: Những loại nội tạng động vật nào được khuyến cáo nên hạn chế để tránh tăng axit uric và hại thận?
Embedding type: token
Reranker: jinaai/jina-reranker-v2-base-multilingual
Metrics: {'hit@1': 1.0, 'hit@5': 1.0, 'recall@5': 1.0, 'mrr@5': 1.0, 'ndcg@5': 1.0}
Top 1 score: 0.72265625
Top 1 text: Tiêu đề: Không muốn hại thận, cần hạn chế 4 loại thịt
Mô tả: Purin trong các loại thịt đỏ chuyển hóa thành axit uric khiến thận phải tăng cường hoạt động. Nếu không xử lý kịp, thận sẽ bị ảnh hưởng nặng nề.
Chuyên mục: Sức khỏe
Đoạn nội dung:
Theo Sohu, nhiều người khi nhắc đến tăng axit uric, gout h


In [19]:
!python src/re-ranker/JINA_TOKEN/jina_rerank_token.py \
    --input "/content/drive/MyDrive/HỌC TẬP/Text mining/out_embedding/per_query_token.jsonl" \
    --output "/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina.jsonl" \
    --batch-size 8 \
    --max-length 1024

--> Đang nạp Tokenizer cho chiến thuật Token...
--> Đang nạp Jina Reranker V2 với cấu hình ép kiểu dữ liệu nửa chính xác float16...
--> Khởi tạo hoàn tất! Thiết bị xử lý hiện hành: CUDA
Reranked 10 queries
Reranked 20 queries
Reranked 30 queries
Reranked 40 queries
Reranked 50 queries
Reranked 60 queries
Reranked 70 queries
Reranked 80 queries
Reranked 90 queries
Reranked 100 queries
Reranked 110 queries
Reranked 120 queries
Reranked 130 queries
Reranked 140 queries
Reranked 150 queries
Done. Wrote 152 rows to /content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina.jsonl


In [20]:
from pathlib import Path

input_file = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_embedding/per_query_token.jsonl")
output_file = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina.jsonl")

def count_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        return sum(1 for line in f if line.strip())

print("Input lines:", count_jsonl(input_file))
print("Output lines:", count_jsonl(output_file))

Input lines: 152
Output lines: 152


In [21]:
import json
import pandas as pd
from pathlib import Path

output_file = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina.jsonl")
summary_file = Path("/content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina_summary.csv")

rows = []

with output_file.open("r", encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        rows.append(row["rerank_metrics"])

df = pd.DataFrame(rows)

summary = {
    "config": "token_jina_reranker",
    "num_queries": len(df),
    "hit@1": df["hit@1"].mean(),
    "hit@5": df["hit@5"].mean(),
    "recall@5": df["recall@5"].mean(),
    "mrr@5": df["mrr@5"].mean(),
    "ndcg@5": df["ndcg@5"].mean(),
}

summary_df = pd.DataFrame([summary])


summary_df.to_csv(summary_file, index=False, encoding="utf-8-sig")
print(f"Saved summary to {summary_file}")
summary_df

Saved summary to /content/drive/MyDrive/HỌC TẬP/Text mining/out_reranker/rerank_token_jina_summary.csv


,config,num_queries,hit@1,hit@5,recall@5,mrr@5,ndcg@5
0,token_jina_reranker,152,0.809211,0.907895,0.838816,0.854386,0.817722
